# [LAB-08] PBT - EDAㅣ4-다변량 분석(실습코드).ipynb

## #01. 준비작업 

### 1. 이전 분석 내용 가져오기 

In [1]:
%%capture cap

%run "./[LAB-08] PBT - EDAㅣ1-EDA 시작하기(연습문제) yeonju.ipynb"

### 2. 불러온 내용 확인 

In [2]:
print("종속변수:", target) 
print("종속변수 유형: " + ("연속형" if target_is_continuous else "명목형"))
print("연속형 : ", continuous_cols)
print("명목형 : ", nominal_cols)

display(df.head())
display(desc.head())
display(cat_desc.head())

종속변수: charges
종속변수 유형: 연속형
연속형 :  ['age', 'bmi', 'children']
명목형 :  ['sex', 'smoker', 'region']


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.924
1,18,male,33.770,1,no,southeast,1725.552
2,28,male,33.000,3,no,southeast,4449.462
3,33,male,22.705,0,no,northwest,21984.471
4,32,male,28.880,0,no,northwest,3866.855


,count,mean,std,min,25%,50%,75%,max,rel_diff,rdiff_flag,iqr,upper_bound,lower_bound,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio,skew,skew_interpret,kurt,kurt_interpret,log_need
age,1337.000,39.222,14.044,18.000,27.000,39.000,51.000,64.000,0.006,similar,24.000,87.000,-9.000,0,0.000,0,0.000,0,0.000,0.055,symmetric,-1.244,platykurtic,none
bmi,1337.000,30.663,6.100,15.960,26.290,30.400,34.700,53.130,0.009,similar,8.410,47.315,13.675,9,0.007,0,0.000,9,0.007,0.284,symmetric,-0.053,platykurtic,none
children,1337.000,1.096,1.206,0.000,0.000,1.000,2.000,5.000,0.096,similar,2.000,5.000,-3.000,0,0.000,0,0.000,0,0.000,0.937,right tail,0.201,leptokurtic,log1p
charges,1337.000,13279.121,12110.360,1121.874,4746.344,9386.161,16657.717,63770.428,0.415,diff,11911.373,34524.778,-13120.716,139,0.104,0,0.000,139,0.104,1.515,right tail,1.604,leptokurtic,log1p


,sex,smoker,region
count,1337,1337,1337
unique,2,2,4
top,male,no,southeast
freq,675,1063,364


### 3. 라이브러리 참조 

In [3]:
from IPython.display import display, Markdown
from pandas import concat, merge
import numpy as np

In [4]:
# 연속형 독립변수간의 상관분석 
# -> 상관행렬 히트맵과 PairPlot을 출력하고 상관분석 결과를 리턴한다. 
corr = my_stats.multi_correlation(origin, columns= continuous_cols, 
                                  plot=True, reg=True, width=4800, height=4800)

display(corr)

,age,bmi,children
age,1.000,0.108,0.056
bmi,0.108,1.000,0.016
children,0.056,0.016,1.000


method  coef  p-value strength  normality_x  normality_y  \
x   y                                                                      
age bmi       Spearman 0.108    0.000     Weak        False        False   
    children  Spearman 0.056    0.041     Weak        False        False   
bmi children  Spearman 0.016    0.568     Weak        False        False   

              linearity  influential_outlier  high_skew  
x   y                                                    
age bmi            True                False      False  
    children      False                False      False  
bmi children       True                False      False

## #03. 상관분석 결고표 정리 

### 1. 결과표를 양방향으로 만들기 

In [5]:
pairs = corr.reset_index()

# 인덱스를 해제한 원본 상관분석 결과표와 x, y를 바꾼 상관분석 결과표를 결합
pairs = concat([pairs, pairs.rename(columns={'x':'y', 'y':'x'})])

# 상관계수의 절댓값이 큰 순으로 정렬 
pairs.sort_values(by='coef', key=abs, ascending=False, inplace=True)

pairs.head(10)

,x,y,method,coef,p-value,strength,normality_x,normality_y,linearity,influential_outlier,high_skew
0,age,bmi,Spearman,0.108,0.000,Weak,False,False,True,False,False
0,bmi,age,Spearman,0.108,0.000,Weak,False,False,True,False,False
1,age,children,Spearman,0.056,0.041,Weak,False,False,False,False,False
1,children,age,Spearman,0.056,0.041,Weak,False,False,False,False,False
2,bmi,children,Spearman,0.016,0.568,Weak,False,False,True,False,False
2,children,bmi,Spearman,0.016,0.568,Weak,False,False,True,False,False


### 2. 변수별 집계표 

In [6]:
# 앞 단계에서 상관계수에 대해 내림차순 정렬되어있는 상태로 
# x별로 그룹화하여 가장 첫번째 항목 선택 --> 가장 상관관계가 큰 항목 
summary = pairs.groupby('x').first()

# 컬럼이름 수정 
summary.rename(columns={'y':'max-y', 'coef':'max-coef'}, inplace=True)
summary

,max-y,method,max-coef,p-value,strength,normality_x,normality_y,linearity,influential_outlier,high_skew
x,,,,,,,,,,
age,bmi,Spearman,0.108,0.000,Weak,False,False,True,False,False
bmi,age,Spearman,0.108,0.000,Weak,False,False,True,False,False
children,age,Spearman,0.056,0.041,Weak,False,False,False,False,False


### 3. 상관정도가 강한 쌍에 대한 집계 

In [7]:
# 양방향 결과표에서 상관정도가 강한 항목만 추출 
strong = pairs[pairs['strength'] == 'Strong']

# 추출된 결과에서 x별로 그룹화하여 y의 갯수와 y를 콤마로 연결한 문자열 집계
group_pairs = strong.groupby('x')['y'].agg(['count', ', '.join])
group_pairs

,count,join
x,,


### 4. 상관분석 결과표 정리 

In [8]:
# 집계표 x에 대한 상관정도가 가장 큰 y이름과 상관계수 추출
s = summary[['max-y', 'max-coef']]

# "추출된 표"를 기준으로 "상관 정도가 강한 쌍에 대한 집계표" 병합
corr_table = merge(s, group_pairs, left_index=True, right_index=True, how='left')

# 상관정도가 강한 변수가 없을 경우 갯수는 0, y는 "-"로 표시 
corr_table.fillna({'count':0, 'join':"-"}, inplace=True)

# 상관정도가 강한 변수의 갯수와 상관계수의 절대값이 큰 순으로 정렬
corr_table.sort_values(by=['count', 'max-coef'], key=abs,ascending=False, inplace=True)

# 컬럼 이름 수정 
corr_table.rename(columns={'join':'columns'}, inplace=True)

corr_table

,max-y,max-coef,count,columns
x,,,,
age,bmi,0.108,0.000,-
bmi,age,0.108,0.000,-
children,age,0.056,0.000,-


## #04. 상관분석 결과표 정리 모듈화 기능 확인 

### 1. 다중 공선성 신호 감지 

In [9]:
my_stats.correlation_summary(corr)

,max-y,max-coef,count,columns
x,,,,
age,bmi,0.108,0,-
bmi,age,0.108,0,-
children,age,0.056,0,-
